# FRED real-world hallucination evaluation

This Kaggle notebook evaluates the Hugging Face FRED checkpoints on **previously unseen financial questions from real financial documents**. It does not inject synthetic errors. Instead, a separate base model generates a candidate answer, and FRED must decide whether to preserve, edit, or abstain using the supplied evidence.

The default run evaluates 50 examples. Set `N_EXAMPLES = 100` for the larger smoke test. Results are saved under `/kaggle/working/fred_real_world_eval/`.

**Interpretation:** FinQA/TAT-QA contain real financial reports, but the hallucinations in this experiment are naturally generated by the base model. This is a small external generalization check, not a claim of broad production reliability.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes peft datasets sentencepiece pandas scikit-learn tqdm

In [ ]:
import gc
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from peft import PeftModel
except ImportError:
    PeftModel = None

OUTPUT_DIR = Path('/kaggle/working/fred_real_world_eval')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
N_EXAMPLES = int(os.environ.get('FRED_N_EXAMPLES', '50'))
SEED = 42
MAX_NEW_TOKENS = 256
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| examples:', N_EXAMPLES)
assert torch.cuda.is_available(), 'Enable a Kaggle GPU: Settings -> Accelerator -> GPU.'

## 1. Load real financial QA examples

The loader first tries the Hugging Face FinQA mirror and then TAT-QA. You can also upload a JSON/JSONL file to Kaggle and set `LOCAL_DATASET_PATH` below. The normalizer accepts common FinQA/TAT-QA field names.

In [ ]:
LOCAL_DATASET_PATH = ''  # Optional: /kaggle/input/<dataset>/test.json or .jsonl
DATASET_CANDIDATES = [
    ('ibm/finqa', 'test'),
    ('dreamerdeo/finqa', 'test'),
    ('next-tatqa/tatqa', 'test'),
]

def read_local_records(path):
    path = Path(path)
    if path.suffix.lower() == '.jsonl':
        return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    obj = json.loads(path.read_text())
    if isinstance(obj, dict):
        for key in ('data', 'test', 'examples', 'records'):
            if isinstance(obj.get(key), list):
                return obj[key]
    return obj if isinstance(obj, list) else []

def first_value(row, names, default=''):
    for name in names:
        value = row.get(name)
        if value not in (None, '', []):
            return value
    return default

def render_context(row):
    pre = first_value(row, ['pre_text', 'pre_texts', 'context', 'documents'], '')
    post = first_value(row, ['post_text', 'post_texts'], '')
    if isinstance(pre, list): pre = ' '.join(map(str, pre))
    if isinstance(post, list): post = ' '.join(map(str, post))
    return f'{pre}\n{post}'.strip()

def render_table(row):
    table = first_value(row, ['table', 'table_data'], [])
    if not table: return '[No table provided]'
    return '\n'.join(' | '.join(map(str, r)) if isinstance(r, list) else str(r) for r in table)

def normalize_row(row, index, source):
    answer = first_value(row, ['answer', 'gold_answer', 'target', 'gold'], '')
    if isinstance(answer, list): answer = '; '.join(map(str, answer))
    return {
        'id': str(first_value(row, ['id', 'uid', 'question_id'], f'{source}_{index}')),
        'source': source,
        'question': str(first_value(row, ['question', 'query'], '')).strip(),
        'context': render_context(row),
        'table': render_table(row),
        'gold_answer': str(answer).strip(),
    }

def load_real_examples(limit):
    if LOCAL_DATASET_PATH:
        rows = read_local_records(LOCAL_DATASET_PATH)
        source = Path(LOCAL_DATASET_PATH).stem
        records = [normalize_row(r, i, source) for i, r in enumerate(rows)]
    else:
        records = []
        errors = []
        for name, split in DATASET_CANDIDATES:
            try:
                ds = load_dataset(name, split=split, trust_remote_code=True)
                records = [normalize_row(dict(r), i, name) for i, r in enumerate(ds)]
                print('loaded:', name, split, len(records))
                break
            except Exception as exc:
                errors.append(f'{name}: {exc}')
        if not records:
            raise RuntimeError('No dataset loaded. Upload a FinQA/TAT-QA JSON/JSONL file and set LOCAL_DATASET_PATH.\n' + '\n'.join(errors))
    records = [r for r in records if r['question'] and r['context'] and r['gold_answer']]
    rng = np.random.default_rng(SEED)
    rng.shuffle(records)
    return records[:limit]

records = load_real_examples(N_EXAMPLES)
print('usable examples:', len(records))
pd.DataFrame(records)[['id', 'source', 'question', 'gold_answer']].head(3)

## 2. Load models from Hugging Face

Models are loaded one at a time in 4-bit mode to fit a Kaggle GPU. The two Stage 2 checkpoints use the structured JSON prompt. The 7B LoRA checkpoint is evaluated with the legacy XML prompt used by the repository's paper-aligned evaluation.

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
MODEL_SPECS = {
    'fred_qwen3_4b': {'repo': 'Akash-Purbhi/fredphase2-qwen3-4b', 'mode': 'stage2'},
    'fred_qwen25_3b': {'repo': 'Akash-Purbhi/fredphase2-qwen2.5-3b', 'mode': 'stage2'},
    'fred_qwen25_7b_lora': {'repo': 'Akash-Purbhi/fred-qwen2.5-7b-lora', 'mode': 'legacy'},
}

def quant_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

def load_checkpoint(repo, lora=False):
    base_repo = BASE_MODEL if lora else repo
    tokenizer = AutoTokenizer.from_pretrained(repo if not lora else base_repo, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        base_repo, device_map='auto', quantization_config=quant_config(),
        torch_dtype=torch.float16, trust_remote_code=True,
    )
    if lora:
        if PeftModel is None: raise RuntimeError('Install peft to load the LoRA checkpoint.')
        model = PeftModel.from_pretrained(model, repo)
    model.eval()
    return tokenizer, model

def unload(tokenizer, model):
    del tokenizer, model
    gc.collect()
    torch.cuda.empty_cache()

## 3. Generate natural candidate answers with the base model

In [ ]:
def generate(tokenizer, model, prompt):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=4096).to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=0.0, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def answer_prompt(r):
    return f'''You are a financial question-answering model. Answer the question using only the supplied financial evidence. Show a concise answer and calculation when needed.

Context:
{r['context']}

Table:
{r['table']}

Question: {r['question']}
Answer:'''

tok, mdl = load_checkpoint(BASE_MODEL, lora=False)
for r in tqdm(records, desc='Generating base answers'):
    r['candidate_answer'] = generate(tok, mdl, answer_prompt(r))
unload(tok, mdl)
with open(OUTPUT_DIR / 'base_answers.jsonl', 'w') as f:
    for r in records: f.write(json.dumps(r, ensure_ascii=False) + '\n')
pd.DataFrame(records)[['question', 'gold_answer', 'candidate_answer']].head(5)

## 4. Run FRED verification

In [ ]:
def stage2_prompt(r):
    return f'''Context: {r['context']}\n\nTable:\n{r['table']}\n\nQuestion: {r['question']}\n\nResponse to Verify: {r['candidate_answer']}\n\nReturn JSON only with keys action, error_type, reasoning, gold_answer, and tagged_xml. Use one action: preserve, edit, abstain, flag_conflict, request_evidence. If editing, tagged_xml must contain the complete corrected answer with <numerical><delete>wrong</delete><mark>correct</mark></numerical> or <temporal> tags.'''

def legacy_prompt(r):
    return f'''Given the financial evidence below, identify numerical or temporal errors in the response. Add <numerical> or <temporal> tags containing <delete>wrong</delete><mark>correct</mark>. If there is no error, return the response unchanged.\n\nEvidence:\n{r['context']}\n\nTable:\n{r['table']}\n\nQuestion: {r['question']}\n\nResponse: {r['candidate_answer']}\n\nReturn only the edited response.'''

def parse_json(text):
    text = re.sub(r'```(?:json)?', '', text, flags=re.I).replace('```', '').strip()
    match = re.search(r'\{.*\}', text, flags=re.S)
    if not match: return {}
    try: return json.loads(match.group(0))
    except json.JSONDecodeError: return {}

def run_fred(name, spec):
    lora = spec['mode'] == 'legacy'
    tok, mdl = load_checkpoint(spec['repo'], lora=lora)
    for r in tqdm(records, desc=name):
        raw = generate(tok, mdl, stage2_prompt(r) if not lora else legacy_prompt(r))
        parsed = parse_json(raw) if not lora else {'raw_legacy': raw}
        r.setdefault('fred', {})[name] = {'raw': raw, **parsed}
    unload(tok, mdl)

for model_name, spec in MODEL_SPECS.items():
    run_fred(model_name, spec)

with open(OUTPUT_DIR / 'all_predictions.jsonl', 'w') as f:
    for r in records: f.write(json.dumps(r, ensure_ascii=False) + '\n')

## 5. Evaluation metrics

The gold answer is used only for offline scoring. Because FinQA/TAT-QA answers may have harmless wording differences, exact string match is supplemented with normalized numeric matching. Manual review is still required for ambiguous cases.

In [ ]:
def norm_text(x):
    return re.sub(r'[^a-z0-9.%$-]+', ' ', str(x).lower()).strip()

def numbers(x):
    return [float(v.replace(',', '')) for v in re.findall(r'-?\d+(?:,\d{3})*(?:\.\d+)?', str(x))]

def answer_matches(pred, gold):
    if not pred or not gold: return False
    if norm_text(pred) == norm_text(gold): return True
    pn, gn = numbers(pred), numbers(gold)
    return bool(pn and gn and any(abs(a-b) <= max(1e-3, abs(b)*0.01) for a in pn for b in gn))

def corrected_answer(text):
    text = re.sub(r'<(?:numerical|temporal|relation)>\s*<delete>.*?</delete>\s*<mark>(.*?)</mark>\s*</(?:numerical|temporal|relation)>', r'\1', str(text), flags=re.S | re.I)
    text = re.sub(r'<[^>]+>', '', text)
    return text.strip()

def has_edit_xml(text):
    return bool(re.search(r'<(numerical|temporal|relation)>.*?<delete>.+?</delete>.*?<mark>.+?</mark>.*?</\1>', str(text), re.S))

def score_model(name, rows):
    stage2 = name != 'fred_qwen25_7b_lora'
    counts = {'n': len(rows), 'naturally_wrong': 0, 'detected_wrong': 0, 'missed_wrong': 0, 'correct_preserved': 0, 'false_correction': 0, 'valid_edit': 0, 'corrected': 0, 'abstained': 0}
    details = []
    for r in rows:
        candidate_ok = answer_matches(r['candidate_answer'], r['gold_answer'])
        naturally_wrong = not candidate_ok
        p = r['fred'][name]
        action = str(p.get('action', '')).lower() if stage2 else ('edit' if has_edit_xml(p.get('raw_legacy', '')) else 'preserve')
        corrected_text = p.get('tagged_xml', '') if stage2 else p.get('raw_legacy', '')
        detected = action in {'edit', 'abstain', 'flag_conflict', 'request_evidence'}
        if naturally_wrong: counts['naturally_wrong'] += 1
        if naturally_wrong and detected: counts['detected_wrong'] += 1
        if naturally_wrong and not detected: counts['missed_wrong'] += 1
        if not naturally_wrong and action == 'preserve': counts['correct_preserved'] += 1
        if not naturally_wrong and action == 'edit': counts['false_correction'] += 1
        if action == 'abstain': counts['abstained'] += 1
        if action == 'edit' and (not stage2 or has_edit_xml(corrected_text)): counts['valid_edit'] += 1
        if naturally_wrong and action == 'edit' and answer_matches(corrected_answer(corrected_text), r['gold_answer']): counts['corrected'] += 1
        details.append({'id': r['id'], 'candidate_ok': candidate_ok, 'action': action, 'candidate': r['candidate_answer'], 'gold': r['gold_answer'], 'fred_output': p.get('raw', p.get('raw_legacy', ''))})
    nwrong = counts['naturally_wrong']; nclean = counts['n'] - nwrong
    counts.update({'natural_hallucination_rate_pct': round(100*nwrong/counts['n'], 2), 'detection_recall_pct': round(100*counts['detected_wrong']/nwrong, 2) if nwrong else 0.0, 'clean_preservation_pct': round(100*counts['correct_preserved']/nclean, 2) if nclean else 0.0, 'false_correction_pct': round(100*counts['false_correction']/nclean, 2) if nclean else 0.0, 'correction_success_pct': round(100*counts['corrected']/nwrong, 2) if nwrong else 0.0, 'valid_edit_pct': round(100*counts['valid_edit']/max(1, sum(1 for d in details if d['action']=='edit')), 2)})
    return counts, details

summaries, detail_rows = [], {}
for name in MODEL_SPECS:
    summary, details = score_model(name, records)
    summaries.append({'model': name, **summary})
    detail_rows[name] = details

summary_df = pd.DataFrame(summaries)
display(summary_df.T)
summary_df.to_csv(OUTPUT_DIR / 'metrics_summary.csv', index=False)
with open(OUTPUT_DIR / 'metrics_details.json', 'w') as f: json.dump(detail_rows, f, indent=2, ensure_ascii=False)

## 6. Manual review sheet and conclusions

Open `manual_review.csv` after the run. Review every row where FRED disagrees with the evidence, especially numeric units, dates, and answers that contain several numbers. Record `human_label` as `correct`, `incorrect`, or `ambiguous`; this produces a defensible small-scale external evaluation.

In [ ]:
review = pd.DataFrame([{
    'id': r['id'], 'source': r['source'], 'question': r['question'],
    'context': r['context'], 'table': r['table'], 'gold_answer': r['gold_answer'],
    'candidate_answer': r['candidate_answer'],
    **{f'{name}_action': r['fred'][name].get('action', 'legacy_xml') for name in MODEL_SPECS},
    'human_label': '', 'notes': ''
} for r in records])
review.to_csv(OUTPUT_DIR / 'manual_review.csv', index=False)
print('Saved:', OUTPUT_DIR)
print('Use these conclusions:')
print('- Synthetic results measure controlled-error performance.')
print('- This notebook measures generalization to naturally generated answers over real financial evidence.')
print('- Report counts and percentages, and manually review ambiguous cases before making claims.')